<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/4%20-%20Redes%20Neurais/4.5_otimizadores_comparacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 4.5 — Comparação de otimizadores numa rede neural

No notebook 4.4 vimos o caminho de cada otimizador no plano dos dois parâmetros de uma regressão linear. Numa rede com camada escondida há dezenas de parâmetros e não dá mais para desenhar o mapa da função de custo. O que dá para comparar é a **curva de custo**: quanto o custo cai a cada passo com cada otimizador.

O problema é ajustar uma curva $y = x^2$ com ruído, que uma reta não consegue representar, usando uma rede com uma camada escondida de 20 neurônios. Treinamos cinco cópias da mesma rede, uma por otimizador, com os mesmos mini-batches.

## Objetivos

- Treinar cinco cópias idênticas de uma rede, cada uma com um otimizador (SGD, momentum, Nesterov, RMSprop e Adam), e comparar as curvas de custo.
- Garantir uma comparação justa: mesma inicialização, mesmos mini-batches, mesma taxa de aprendizado.
- Verificar se a ordem dos otimizadores muda quando mudamos a taxa de aprendizado.

Referências: S. Ruder, *An overview of gradient descent optimization algorithms* (https://www.ruder.io/optimizing-gradient-descent/); documentação de `torch.optim` (https://pytorch.org/docs/stable/optim.html). O exemplo foi inspirado nos tutoriais de Morvan Zhou (https://mofanpy.com/tutorials/).

## Resumo das regras de atualização

As equações foram apresentadas, com mais detalhes, no notebook 4.4. Notação: $W_t$ são os parâmetros no passo $t$ (uma atualização por mini-batch), $g_t = \nabla J(W_t)$ é o gradiente da função de custo (loss) calculado no mini-batch, $\eta$ é a taxa de aprendizado (learning rate, argumento `lr`) e $\epsilon$ é uma constante pequena que evita divisão por zero. Nos métodos adaptativos, as operações são feitas elemento a elemento (cada parâmetro tem o seu passo). As fórmulas seguem a implementação do PyTorch.

| Método | Regra de atualização | Hiperparâmetros (neste notebook) |
|---|---|---|
| SGD | $W_{t+1} = W_t - \eta \, g_t$ | `lr` |
| Momentum | $v_{t+1} = \gamma v_t + g_t$, $\quad W_{t+1} = W_t - \eta \, v_{t+1}$ | $\gamma$ = `momentum` = 0,9 |
| Nesterov | $v_{t+1} = \gamma v_t + g_t$, $\quad W_{t+1} = W_t - \eta \, (g_t + \gamma v_{t+1})$ | $\gamma$ = `momentum` = 0,9, `nesterov=True` |
| Adagrad | $G_t = G_{t-1} + g_t^2$, $\quad W_{t+1} = W_t - \eta \, g_t / (\sqrt{G_t} + \epsilon)$ | — (usado no exercício 1) |
| RMSprop | $s_t = \alpha s_{t-1} + (1-\alpha) g_t^2$, $\quad W_{t+1} = W_t - \eta \, g_t / (\sqrt{s_t} + \epsilon)$ | $\alpha$ = `alpha` = 0,9 |
| Adam | $m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$, $\quad s_t = \beta_2 s_{t-1} + (1-\beta_2) g_t^2$, $\quad W_{t+1} = W_t - \eta \, \hat{m}_t / (\sqrt{\hat{s}_t} + \epsilon)$ | $(\beta_1, \beta_2)$ = `betas` = (0,9; 0,99) |

- $v_t$: velocidade (acúmulo dos gradientes) do momentum, com $v_0 = 0$.
- $G_t$: soma dos quadrados de todos os gradientes desde o início (Adagrad); $s_t$: média móvel exponencial dos quadrados dos gradientes (RMSprop e Adam).
- $m_t$: média móvel exponencial dos gradientes (Adam); $\hat{m}_t = m_t / (1 - \beta_1^t)$ e $\hat{s}_t = s_t / (1 - \beta_2^t)$ são as médias com correção de viés, porque $m_0 = s_0 = 0$.

## Importação das bibliotecas

In [ ]:
import copy

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torch.utils.data

torch.manual_seed(1);    # semente do gerador pseudoaleatório

Hiperparâmetros do experimento. `REGR_RN` escolhe o problema: `False` usa a curva $y = x^2$ e a rede com camada escondida; `True` usa uma reta e uma regressão linear (um neurônio), como no 4.4.

In [ ]:
lr = 0.01
batch_size = 50
n_epochs = 6
REGR_RN = False  # True: regressão linear; False: rede neural com 2 camadas

## Dados sintéticos

500 valores de $x$ igualmente espaçados em $[-1, 1]$. `torch.unsqueeze(..., dim=1)` transforma o vetor de shape (500,) numa coluna de shape (500, 1): uma amostra por linha e um atributo por coluna, o formato que `nn.Linear` espera. O ruído é gaussiano, com desvio-padrão 0,1 (`torch.randn` sorteia valores com média 0 e desvio 1).

In [ ]:
x = torch.unsqueeze(torch.linspace(-1, 1, 500), dim=1)   # shape (500, 1)
if REGR_RN:
    y = 0.25 * x + 0.68 + 0.1 * torch.randn(x.size())
else:
    y = x.pow(2) + 0.1 * torch.randn(x.size())
print('x:', x.shape, ' y:', y.shape)

plt.scatter(x.numpy(), y.numpy(), s=5)
plt.xlabel('x')
plt.ylabel('y')
plt.show()

## Dataset e DataLoader

Com 500 amostras e mini-batches de 50, cada época tem 10 iterações; em 6 épocas são 60 atualizações dos parâmetros de cada rede.

In [ ]:
torch_dataset = torch.utils.data.TensorDataset(x, y)
loader = torch.utils.data.DataLoader(
    dataset=torch_dataset,
    batch_size=batch_size,
    shuffle=True)
print('iterações por época:', len(loader))

## Modelos

`Net1` é a regressão linear (um neurônio, 2 parâmetros). `Net2` tem uma camada escondida de 20 neurônios com ativação ReLU e uma saída linear: $20 + 20$ parâmetros na camada escondida (pesos e bias) e $20 + 1$ na saída, 61 no total. Nas duas, os pesos e o bias da camada de saída começam em zero; os pesos da camada escondida de `Net2` são sorteados pela inicialização padrão do PyTorch.

In [ ]:
class Net1(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.predict = torch.nn.Linear(1, 1)   # saída
        torch.nn.init.constant_(self.predict.weight, 0.)  # peso em 0
        torch.nn.init.constant_(self.predict.bias, 0.)    # bias em 0

    def forward(self, x):
        x = self.predict(x)
        return x

In [ ]:
class Net2(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = torch.nn.Linear(1, 20)    # camada escondida
        self.predict = torch.nn.Linear(20, 1)   # saída
        torch.nn.init.constant_(self.predict.weight, 0.)  # peso em 0
        torch.nn.init.constant_(self.predict.bias, 0.)    # bias em 0

    def forward(self, x):
        x = F.relu(self.hidden(x))      # função de ativação da camada escondida
        x = self.predict(x)             # saída linear
        return x

## Cinco cópias da mesma rede

Para que a comparação seja justa, as cinco redes precisam começar com **os mesmos** parâmetros. Se criássemos cada uma com `Net2()`, os pesos da camada escondida seriam sorteados de novo para cada rede, e parte da diferença entre as curvas viria da inicialização, não do otimizador. Por isso criamos uma rede e fazemos cópias com `copy.deepcopy`, que copia também os tensores dos parâmetros. Guardamos o estado inicial (`state_dict`, o dicionário com os tensores de parâmetros) para usar nos exercícios.

In [ ]:
net_SGD      = Net1() if REGR_RN else Net2()
net_Momentum = copy.deepcopy(net_SGD)
net_Nesterov = copy.deepcopy(net_SGD)
net_RMSprop  = copy.deepcopy(net_SGD)
net_Adam     = copy.deepcopy(net_SGD)
nets = [net_SGD, net_Momentum, net_Nesterov, net_RMSprop, net_Adam]
init_state = copy.deepcopy(net_SGD.state_dict())   # parâmetros iniciais, comuns às cinco redes

print(net_SGD)
print('parâmetros:', sum(p.numel() for p in net_SGD.parameters()))

## Cinco otimizadores, um para cada rede

Cada otimizador recebe os parâmetros de uma das redes e guarda o seu próprio estado (a velocidade $v$ do momentum, as médias $m$ e $s$ do Adam etc.). A função de custo é o MSE (`nn.MSELoss`), a mesma para as cinco.

In [ ]:
opt_SGD      = torch.optim.SGD(net_SGD.parameters(), lr=lr)
opt_Momentum = torch.optim.SGD(net_Momentum.parameters(), lr=lr, momentum=0.9)
opt_Nesterov = torch.optim.SGD(net_Nesterov.parameters(), lr=lr, momentum=0.9, nesterov=True)
opt_RMSprop  = torch.optim.RMSprop(net_RMSprop.parameters(), lr=lr, alpha=0.9)
opt_Adam     = torch.optim.Adam(net_Adam.parameters(), lr=lr, betas=(0.9, 0.99))
optimizers   = [opt_SGD, opt_Momentum, opt_Nesterov, opt_RMSprop, opt_Adam]
labels       = ['SGD', 'Momentum', 'Nesterov', 'RMSprop', 'Adam']

In [ ]:
criterion = torch.nn.MSELoss()
losses_his = [[], [], [], [], []]   # custo de cada passo, por rede
ws_his = [[], [], [], [], []]       # primeiro peso da camada de saída, por rede
bs_his = [[], [], [], [], []]       # bias da camada de saída, por rede

## Treinamento

Para cada mini-batch, as cinco redes recebem os mesmos dados e fazem um passo cada uma. O custo registrado é o do mini-batch, calculado antes da atualização.

In [ ]:
for epoch in range(n_epochs):
    print('Época:', epoch)
    for step, (b_x, b_y) in enumerate(loader):      # para cada mini-batch
        for net, opt, l_his, w_his, b_his in zip(nets, optimizers, losses_his, ws_his, bs_his):
            output = net(b_x)                # predição de cada rede
            loss = criterion(output, b_y)    # custo de cada rede
            opt.zero_grad()                  # zera os gradientes do passo anterior
            loss.backward()                  # retropropagação: calcula os gradientes
            opt.step()                       # aplica a regra de atualização do otimizador
            l_his.append(loss.item())        # .item(): número Python, sem o grafo de gradientes
            w_his.append(net.predict.weight[0, 0].item())
            b_his.append(net.predict.bias[0].item())

In [ ]:
if REGR_RN:
    # regressão linear: caminho dos parâmetros no plano (b, w)
    for i, (w_his, b_his) in enumerate(zip(ws_his, bs_his)):
        plt.plot(b_his, w_his, 'o-', ms=3, label=labels[i])
    plt.xlabel('b (bias)')
    plt.ylabel('w (peso)')
else:
    for i, l_his in enumerate(losses_his):
        plt.plot(l_his, label=labels[i])
    plt.xlabel('Passos (mini-batches)')
    plt.ylabel('Custo (MSE) no mini-batch')
plt.legend(loc='best')
plt.show()

## Avaliação

O custo de um mini-batch oscila conforme as amostras sorteadas. Para comparar o resultado final, calculamos o custo de cada rede no conjunto inteiro e desenhamos a curva que cada uma aprendeu. `torch.no_grad()` desliga o registro de operações para o gradiente, que não é necessário na avaliação.

In [ ]:
with torch.no_grad():
    plt.scatter(x.numpy(), y.numpy(), s=5, c='lightgray', label='dados')
    for net, label in zip(nets, labels):
        pred = net(x)
        print(f'{label:<9} custo final (500 amostras): {criterion(pred, y).item():.4f}')
        plt.plot(x.numpy(), pred.numpy(), lw=2, label=label)
plt.xlabel('x')
plt.ylabel('y')
plt.legend(loc='best')
plt.show()

## Exercícios

1. Acrescente o Adagrad (`torch.optim.Adagrad`, com o mesmo `lr`) à comparação, como sexta rede. Para manter a comparação justa, as seis redes devem partir de `init_state` e ver os mesmos mini-batches; escreva uma função `compare(lr, n_epochs)` que crie as seis redes e os seis otimizadores, treine e devolva as listas de custo, e desenhe as curvas. O exercício 2 usa essa função.

In [ ]:
labels6 = labels + ['Adagrad']

def compare(lr, n_epochs, seed=1):
    # TODO: escreva sua solução aqui
    return his

def plot_compare(his, title):
    for l_his, label in zip(his, labels6):
        plt.plot(l_his, label=label)
    plt.title(title)
    plt.yscale('log')   # escala log: picos grandes e diferenças pequenas no mesmo gráfico
    plt.xlabel('Passos (mini-batches)')
    plt.ylabel('Custo (MSE) no mini-batch, escala log')
    plt.legend(loc='best')
    plt.show()
    for l_his, label in zip(his, labels6):
        print(f'{label:<9} média do custo nos 10 últimos passos: {sum(l_his[-10:]) / 10:.4f}')

his_001 = compare(lr=0.01, n_epochs=n_epochs)
plot_compare(his_001, 'lr = 0.01')

*Escreva sua resposta aqui.*

2. Antes de executar, preveja: se a taxa de aprendizado for 10 vezes maior (`lr=0.1`), a ordem das curvas muda? Execute `compare(lr=0.1, n_epochs=n_epochs)` e verifique. Depois teste também `lr=0.001` com 30 épocas.

In [ ]:
# TODO: escreva sua solução aqui


In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

3. Mude `REGR_RN` para `True` e execute o notebook de novo (Kernel → Reiniciar e executar tudo). O gráfico do treinamento passa a mostrar o caminho dos parâmetros $(b, w)$ da regressão linear. Compare com os caminhos do notebook 4.4: os otimizadores se comportam da mesma forma?

*Escreva sua resposta aqui.*

## Aprendizados

- Comparar otimizadores exige as mesmas condições: mesma inicialização (`copy.deepcopy` ou `load_state_dict`), mesmos mini-batches e o mesmo número de passos.
- O custo de cada mini-batch oscila; para comparar o resultado final, avalie o custo no conjunto inteiro.
- A ordem dos otimizadores depende da taxa de aprendizado: cada método tem uma faixa de `lr` em que funciona bem, e a escolha do otimizador e do `lr` é feita em conjunto.
- Os métodos adaptativos (RMSprop, Adam, Adagrad) ajustam o passo de cada parâmetro a partir do histórico dos gradientes. Uma variante do Adam com decaimento de pesos separado, o AdamW, é o otimizador usado no pré-treino da maioria dos LLMs.